Load packages

In [1]:
import os
import napari
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import tifffile as tiff
import cv2
from collections import defaultdict
from skimage import morphology
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation, feature
from skimage.transform import resize
import re
import shutil
from scipy.ndimage import binary_fill_holes 
from sam2.build_sam import build_sam2_video_predictor
from pathlib import Path

# napari 라이브러리는 별도로 필요할 때만 import합니다
# (Jupyter 환경에서 GUI 라이브러리 초기화가 커널 충돌을 유발할 수 있음)

Select device for computation

In [2]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":

    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

using device: cuda


Load SAM2

In [3]:
# directory for SAM2 packages
sam2_pkg_dir = "C:\Windows\System32\segment-anything-2\sam2"

# save directory for images to be processed
base_dir = "E:\python projects\codes\sam2"

# initiate SAM2 
# directory for checkpoints of SAM2
sam2_checkpoint = os.path.join(base_dir, "checkpoints", "sam2.1_hiera_small.pt")

# directory for config of SAM2 
model_cfg = os.path.join(base_dir,"sam2", "configs", "sam2.1", "sam2.1_hiera_s.yaml")

# generate predictor 
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

In [4]:
import inspect, pydoc

cls = type(predictor)
print("Class:", cls.__module__, cls.__name__)

# 1) __init__ 시그니처
try:
    sig = inspect.signature(cls.__init__)
    print("\n[__init__ signature]")
    print(sig)
except Exception as e:
    print("\n[__init__ signature] failed:", e)

# 2) __init__ 문서 문자열
doc = getattr(cls.__init__, "__doc__", None)
if doc:
    print("\n[__init__ docstring]\n", doc)

# 3) 소스 코드 경로/일부
try:
    src = inspect.getsource(cls.__init__)
    print("\n[__init__ source snippet]\n", src[:2000])  # 길면 앞부분만
except Exception as e:
    try:
        fpath = inspect.getfile(cls)
        print("\n[Source file path]", fpath)
        # 필요하면 파일 열어서 'show'/'visual'/'viewer' 문자열 검색
    except Exception as e2:
        print("\n[Source path lookup failed]", e, e2)


Class: sam2.sam2_video_predictor SAM2VideoPredictor

[__init__ signature]
(self, fill_hole_area=0, non_overlap_masks=False, clear_non_cond_mem_around_input=False, add_all_frames_to_correct_as_cond=False, **kwargs)

[__init__ source snippet]
     def __init__(
        self,
        fill_hole_area=0,
        # whether to apply non-overlapping constraints on the output object masks
        non_overlap_masks=False,
        # whether to clear non-conditioning memory of the surrounding frames (which may contain outdated information) after adding correction clicks;
        # note that this would only apply to *single-object tracking* unless `clear_non_cond_mem_for_multi_obj` is also set to True)
        clear_non_cond_mem_around_input=False,
        # if `add_all_frames_to_correct_as_cond` is True, we also append to the conditioning frame list any frame that receives a later correction click
        # if `add_all_frames_to_correct_as_cond` is False, we conditioning frame list to only use thos

In [5]:
# --- Pre-process  
def preprocess(file_path):
    """
    TIFF 3D 스택을 읽어 다운샘플/등방성 근사 → 3D Sobel 경계강도 → 정규화 → 축 변환.
    반환:
        pre_data : 최종 전처리된 그라디언트 볼륨 (Z,Y,X)
        data     : 리사이즈 및 축 변환된 원본 볼륨 (Z,Y,X)
    """
    # 0) load
    
    data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
    data_path = os.path.join(file_path)

    data = tiff.imread(data_path)
    data = np.array(data)

    if data.ndim != 3:
        raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data.shape}")

    # 1) Resizing: downsampled isotropic resolution (원 코드 그대로)
    dx = 0.1126   # in-plane (x,y) spacing
    dz = 0.5485   # slice spacing (z)

    W1 = data.shape[1]
    H1 = data.shape[2]
    D1 = int(round(data.shape[0] * dz / dx))

    W2 = int(round(0.3 * W1))
    H2 = int(round(0.3 * H1))
    D2 = int(round(0.3 * D1))

    data_size = (D2, W2, H2)  # 원 코드 유지
    data = resize(
        data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
    ).astype(data.dtype, copy=False)

    # 2) 3D gradient (Sobel magnitude) — 원 축 정의 유지
    def imgradient3(V):
        Gx = ndi.sobel(V, axis=1, mode='nearest')  # X(열)
        Gy = ndi.sobel(V, axis=0, mode='nearest')  # Y(행)
        Gz = ndi.sobel(V, axis=2, mode='nearest')  # Z(슬라이스)
        return np.sqrt(Gx*Gx + Gy*Gy + Gz*Gz)

    grad = imgradient3(data)

    # 3) normalize (원 코드 그대로)
    norm_min = np.max(grad) * 0.01
    norm_max = np.max(grad) * 0.2
    grad = np.clip(grad, norm_min, norm_max)
    grad = (grad - norm_min) / (norm_max - norm_min)   # ← normalize to [0,1]

    # 4) permute (원 주석 유지: (X, Y, Z) -> (Z, Y, X))
    data = np.transpose(data, (2, 0, 1))
    grad = np.transpose(grad, (2, 0, 1))

    pre_data = grad
    return pre_data, data   # pre_data : preprocessed HT data; data: HT data

# --- Prepare Manual Slice Labels
def load_labels(file_path, viewer=None):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # 'labels'가 이름에 포함된 파일만 선별
    candidates = [f for f in os.listdir(file_path) if "labels" in f.lower()]
    if not candidates:
            viewer.add_labels(
                np.zeros_like(viewer.layers[0].data, dtype=np.uint8),
                name="Labels"
            )

    for fname in sorted(candidates):
        fpath = os.path.join(file_path, fname)
        lower = fname.lower()

        try:
            if lower.endswith(('.tif', '.tiff')):
                arr = tiff.imread(fpath)
            elif lower.endswith('.npy'):
                arr = np.load(fpath)
            else:
                print(f"⏭️ Skip (unsupported): {fname}")
                continue
            if not np.issubdtype(arr.dtype, np.integer):
                arr = arr.astype(np.int32)          
            # 🔻 확장자 제거한 이름 사용
            layer_name = Path(fname).stem
            viewer.add_labels(arr, name=layer_name)
            print(f"✅ Loaded to napari: {layer_name} (from '{fname}', shape={arr.shape})")

        except Exception as e:
            print(f"❌ Failed to load '{fname}': {e}")

    return viewer

# --- Prepare jpg images for SAM2 operation 
def jpg_for_sam2(file_path, pre_data):

    prefix = os.path.splitext(os.path.basename(file_path))[0]
    # output_dir = os.path.join(file_path, prefix)
    output_dir = r"G:\RealData\sam2"

    os.makedirs(output_dir, exist_ok=True)    

    for i, frame in enumerate(pre_data, start=1):
        norm_frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        uint8_frame = norm_frame.astype(np.uint8)
        rgb_frame = cv2.cvtColor(uint8_frame, cv2.COLOR_GRAY2BGR)
        filename = f"{i:04d}.jpg"
        out_path = os.path.join(output_dir, filename)
        cv2.imwrite(out_path, rgb_frame)

    print(f"✔ {len(pre_data)} slices saved to: {output_dir}")

    # viewer.add_image(pre_data, name='raw', rgb=False)
    
    # napari.run()

    return pre_data, output_dir

# --- SAM2 operation 
def propagate_mask_and_visualize(
    jpg_path,
    predictor,
    label_layer=None,
    obj_id=1,
    seed_planes=None,
    viewer=None
):
    import numpy as np, os, napari
    from napari.layers import Labels as LabelsLayer

    # (0) viewer
    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # (1) label_layer 자동
    if (label_layer is None) or (not hasattr(label_layer, "data")):
        try:
            label_layer = viewer.layers['Labels']
        except KeyError:
            candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
            if not candidates:
                raise RuntimeError(
                    "napari 뷰어에서 Labels 레이어를 찾지 못했습니다. "
                    "viewer.layers['Labels'] 이름을 맞추거나, Labels 레이어를 추가하세요."
                )
            label_layer = candidates[0]
            print(f"⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

    # (2) 데이터
    import numpy as np, os
    label_data = np.asarray(label_layer.data)
    if label_data.ndim != 3:
        raise ValueError(f"Labels 데이터는 (Z, Y, X) 3D여야 합니다. 현재 shape={label_data.shape}")
    num_slices = label_data.shape[0]

    # (A) obj_id 존재 슬라이스
    if seed_planes is None:
        seed_planes = np.where(
            (label_data == obj_id).reshape(num_slices, -1).any(axis=1)
        )[0].tolist()

    if not seed_planes:
        uniq = np.unique(label_data)
        raise ValueError(
            f"[중단] obj_id={obj_id} 라벨이 없습니다. Labels 유니크 값(일부): {uniq[:20]}"
        )

    print(f"[seed check] obj_id={obj_id}, seed_planes (len={len(seed_planes)}): "
          f"{seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

    # (3) SAM2 predictor 초기화
    inference_state = predictor.init_state(video_path=jpg_path)
    predictor.reset_state(inference_state)

    # (B) 씨드 추가
    for fidx in seed_planes:
        manual_mask = (label_data[fidx] == obj_id)
        if manual_mask.any():
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=fidx,
                obj_id=obj_id,
                mask=manual_mask,
            )
            print(f"✔ Added seed @ z={fidx} (px={int(manual_mask.sum())})")
        else:
            print(f"⚠️ z={fidx}에 obj_id={obj_id} 마스크가 비어 있음 → 건너뜀")

    # (4) 전파
    center_seed = seed_planes[len(seed_planes)//2]
    print(f"🔁 Propagating from center z={center_seed}, seeds={len(seed_planes)}")

    video_segments = {}
    for rev in (False, True):
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=center_seed, reverse=rev
        ):
            per_obj_output_mask = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
            if out_frame_idx in video_segments:
                for oid, mask in per_obj_output_mask.items():
                    if oid in video_segments[out_frame_idx]:
                        video_segments[out_frame_idx][oid] |= mask
                    else:
                        video_segments[out_frame_idx][oid] = mask
            else:
                video_segments[out_frame_idx] = per_obj_output_mask

    if len(video_segments) == 0:
        raise RuntimeError("No masks were propagated. Check your seed mask or predictor output.")

    # (5) 라벨 스택 생성
    sample_mask = np.squeeze(next(iter(next(iter(video_segments.values())).values())))
    height, width = sample_mask.shape
    label_stack = np.zeros((num_slices, height, width), dtype=np.uint8)

    for fidx, per_obj_mask in video_segments.items():
        if obj_id in per_obj_mask:
            mask_i = np.squeeze(per_obj_mask[obj_id])
            label_stack[fidx][mask_i > 0] = obj_id

    # # (6) 작업 끝나면 jpg_path 폴더 삭제
    # try:
    #     shutil.rmtree(jpg_path)
    #     print(f"🧹 Removed temporary folder: {jpg_path}")
    # except Exception as e:
    #     print(f"⚠️ Failed to remove {jpg_path}: {e}")

    return label_stack, video_segments

# --- Post-process 
def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix=None,                # 🆕 'ooplasm' / 'inner' / 'outer' / 기타
                     return_as_label_id=True):
    
    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f" obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- preserve the largest object using watershed ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # seed mask
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # --- morphological operation (varies with target)
    s = str(suffix).lower()

    config = {
        'pb':         (3, ('opening', 'closing')),
        'ooplasm':    (5, ('opening', 'closing')),
    }
    size, ops = config.get(s, (5, ('closing', 'opening'))) # default
    print(f"[morphology] suffix='{s}', size={size}, ops={ops}")

    selem = morphology.ball(size)
    for op in ops:
        mask = getattr(morphology, f"binary_{op}")(mask, selem)

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)

# --- Run SAM2 segmentation and post-process for multiple label layers and build a combined 3D mask stack
def process_main(
    jpg_path,
    predictor,
    obj_id=1,
    viewer=None,
    layer_name_keyword="Labels",   # 이 문자열을 포함한 napari Labels 레이어만 대상
    # --- pb 전처리용 외부 의존성 ---
    pre_data=None,                 # 원본/전처리할 볼륨 (Z,Y,X)
    file_path=None,                # jpg_for_sam2 에 전달할 원본 파일 경로
    jpg_for_sam2=None              # callable: (file_path, pre_data) -> (pre_data, output_dir)
):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # --- 대상 레이어 수집 ---
    layers = [lyr for lyr in viewer.layers
              if isinstance(lyr, LabelsLayer) and (layer_name_keyword.lower() in lyr.name.lower())]
    if not layers:
        raise RuntimeError(f"이름에 '{layer_name_keyword}'가 포함된 napari Labels 레이어를 찾지 못했습니다.")

    def extract_suffix(name, fallback_idx):
        # 예: 'Labels(inner)' -> 'inner', 'Labels-outer' -> 'outer'
        m = re.search(r'Labels\s*[\(\-_\s]*([^)]+)[\)]*', name, re.IGNORECASE)
        if m:
            suf = m.group(1).strip()
            suf = re.sub(r'\W+', '_', suf)
            if suf:
                return suf
        return f"labels{fallback_idx}"

    # --- 처리 순서: ooplasm -> inner -> outer -> pb (대소문자 무시) ---
    priority = {"ooplasm": 0, "inner": 1, "outer": 2, "pb": 3}
    indexed = []
    for idx, lyr in enumerate(layers, start=1):
        suf = extract_suffix(lyr.name, idx)
        prio = priority.get(suf.lower(), 999)  # 지정 외 항목은 맨 뒤
        indexed.append((prio, suf, lyr))
    indexed.sort(key=lambda x: (x[0], x[1].lower()))

    predicted_labels = {'per_layer': {}, 'order': []}

    for prio, suffix, lyr in indexed:
        suffix_norm = suffix.lower()
        print(f"\n===== Processing: '{lyr.name}' → suffix='{suffix_norm}' (prio={prio}) =====")
        # pb가 아닌 경우: 기존 파이프라인
        if suffix_norm != "pb":
            label_stack, _ = propagate_mask_and_visualize(
                jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)

        else:
            # pb 전처리 요건 확인
            have_inner = f"final_mask_inner" in globals()
            have_ooplas = f"final_mask_ooplasm" in globals()
            if not (have_inner and have_ooplas):
                print("⚠️ PB 전처리 불가: inner/ooplasm 결과가 필요합니다. PB는 건너뜁니다.")
                continue
            if (pre_data is None) or (file_path is None) or (jpg_for_sam2 is None):
                print("⚠️ PB 전처리 불가: pre_data/file_path/jpg_for_sam2 가 필요합니다. PB는 건너뜁니다.")
                continue

            # (inner & ~ooplasm) 마스크 계산
            inner_mask = (globals()["final_mask_inner"] > 0)
            ooplasm_mask = (globals()["final_mask_ooplasm"] > 0)
            pvs_mask = (inner_mask & ~ooplasm_mask)

            if pvs_mask.shape != pre_data.shape:
                raise ValueError(f"PB 전처리: core_mask shape {pvs_mask.shape} != pre_data shape {pre_data.shape}")

            # segmented pre_data 생성: 마스크 위치만 보존
            if np.issubdtype(pre_data.dtype, np.bool_):
                segmented_pre = pre_data & pvs_mask
            else:
                segmented_pre = (pre_data * pvs_mask.astype(pre_data.dtype))

            # jpg set 재생성
            _pre_data_for_sam2, pb_jpg_path = jpg_for_sam2(file_path, segmented_pre)

            # SAM2 실행 (pb용)
            label_stack, _ = propagate_mask_and_visualize(
                pb_jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)
            # viewer.add_labels(final_mask, name="mask_PB_raw")
            # viewer.add_image(segmented_pre, name="segmented_pre")

        # 개별 결과 저장/노출
        globals()[f"label_stack_{suffix_norm}"] = label_stack
        globals()[f"final_mask_{suffix_norm}"]  = final_mask
        predicted_labels['per_layer'][suffix_norm] = dict(label_stack=label_stack, final_mask=final_mask)
        predicted_labels['order'].append(suffix_norm)

    print("\n✅ 개별 마스크 생성 완료(처리 순서):", predicted_labels['order'])

    for k in list(globals().keys()):
        if k.startswith("final_mask_"):
            norm = k.lower()
            if norm not in globals():
                globals()[norm] = globals()[k]

    # --- 4개 마스크를 통합한 label stack 생성 ---
    need = ["ooplasm", "inner", "outer", "pb"]
    if all(f"final_mask_{n}" in globals() for n in need):
        ooplasm = (globals()["final_mask_ooplasm"] > 0)
        inner   = (globals()["final_mask_inner"]   > 0)
        outer   = (globals()["final_mask_outer"]   > 0)
        pb      = (globals()["final_mask_pb"]      > 0)

        label_stack_all = np.zeros_like(ooplasm, np.uint8)
        # 우선순위: ooplasm(1) → inner-only(2) → outer-only(3) → pb-only(4)
        label_stack_all[ooplasm] = 1
        label_stack_all[inner & ~ooplasm] = 2
        label_stack_all[outer & ~(ooplasm | inner)] = 3
        label_stack_all[pb] = 4

        globals()["label_stack_all"] = label_stack_all
        viewer.add_labels(label_stack_all, name="mask_ALL")
        print("✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)")
        predicted_labels["label_stack_all"] = label_stack_all
    else:
        missing = [n for n in need if f"final_mask_{n}" not in globals()]
        print(f"⚠️ 통합 레이블(label_stack_all) 생성 불가: 부족한 마스크 = {missing}")

    return predicted_labels



In [ ]:
# --- Set file directory
file_path = r"G:\RealData\260129\D1_dish02_05.tiff"
# --- Preprocess
pre_data, data = preprocess(file_path)
pre_data, jpg_path = jpg_for_sam2(file_path, pre_data)

# --- Run viewer: preprae preprocessed images 
viewer = napari.Viewer()
viewer.add_image(pre_data, name = 'preprocessed')
viewer.add_image(data, name = 'HT(downsampled)')

# # --- Load labels: one may generate labels if labels = None
# load_labels(file_path, viewer) # one may generate labels + .tif라는 거 이름에서 제거



multi-compartment segmentation

In [7]:
#multi compartment pre-process
# --- Pre-process  
def preprocess(file_path):
    """
    TIFF 3D 스택을 읽어 다운샘플/등방성 근사 → 3D Sobel 경계강도 → 정규화 → 축 변환.
    반환:
        pre_data : 최종 전처리된 그라디언트 볼륨 (Z,Y,X)
        data     : 리사이즈 및 축 변환된 원본 볼륨 (Z,Y,X)
    """
    # 0) load
    
    data_name = os.path.basename(file_path.rstrip("/\\"))  # → 'cc'
    data_path = os.path.join(file_path)

    data = tiff.imread(data_path)
    data = np.array(data)

    if data.ndim != 3:
        raise ValueError(f"3D TIFF가 필요합니다. 현재 shape={data.shape}")

    # 1) Resizing: downsampled isotropic resolution (원 코드 그대로)
    dx = 0.1126   # in-plane (x,y) spacing
    dz = 0.5485   # slice spacing (z)

    W1 = data.shape[1]
    H1 = data.shape[2]
    D1 = int(round(data.shape[0] * dz / dx))

    W2 = int(round(0.3 * W1))
    H2 = int(round(0.3 * H1))
    D2 = int(round(0.3 * D1))

    data_size = (D2, W2, H2)  # 원 코드 유지
    data = resize(
        data, data_size, order=1, mode='edge', anti_aliasing=False, preserve_range=True
    ).astype(data.dtype, copy=False)

    # 2) 3D gradient (Sobel magnitude) — 원 축 정의 유지
    def imgradient3(V):
        Gx = ndi.sobel(V, axis=1, mode='nearest')  # X(열)
        Gy = ndi.sobel(V, axis=0, mode='nearest')  # Y(행)
        Gz = ndi.sobel(V, axis=2, mode='nearest')  # Z(슬라이스)
        return np.sqrt(Gx*Gx + Gy*Gy + Gz*Gz)

    grad = imgradient3(data)

    # 3) normalize (원 코드 그대로)
    norm_min = np.max(grad) * 0.01
    norm_max = np.max(grad) * 0.2
    grad = np.clip(grad, norm_min, norm_max)
    grad = (grad - norm_min) / (norm_max - norm_min)   # ← normalize to [0,1]

    # 4) permute (원 주석 유지: (X, Y, Z) -> (Z, Y, X))
    data = np.transpose(data, (2, 0, 1))
    grad = np.transpose(grad, (2, 0, 1))

    pre_data = grad
    return pre_data, data   # pre_data : preprocessed HT data; data: HT data

# --- Prepare Manual Slice Labels
def load_labels(file_path, viewer=None):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # 'labels'가 이름에 포함된 파일만 선별
    candidates = [f for f in os.listdir(file_path) if "labels" in f.lower()]
    if not candidates:
            viewer.add_labels(
                np.zeros_like(viewer.layers[0].data, dtype=np.uint8),
                name="Labels"
            )

    for fname in sorted(candidates):
        fpath = os.path.join(file_path, fname)
        lower = fname.lower()

        try:
            if lower.endswith(('.tif', '.tiff')):
                arr = tiff.imread(fpath)
            elif lower.endswith('.npy'):
                arr = np.load(fpath)
            else:
                print(f"⏭️ Skip (unsupported): {fname}")
                continue
            if not np.issubdtype(arr.dtype, np.integer):
                arr = arr.astype(np.int32)          
            # 🔻 확장자 제거한 이름 사용
            layer_name = Path(fname).stem
            viewer.add_labels(arr, name=layer_name)
            print(f"✅ Loaded to napari: {layer_name} (from '{fname}', shape={arr.shape})")

        except Exception as e:
            print(f"❌ Failed to load '{fname}': {e}")

    return viewer

# --- Prepare jpg images for SAM2 operation 
def jpg_for_sam2(file_path, pre_data):

    prefix = os.path.splitext(os.path.basename(file_path))[0]
    # output_dir = os.path.join(file_path, prefix)
    output_dir = r"G:\RealData\sam2"

    os.makedirs(output_dir, exist_ok=True)    

    for i, frame in enumerate(pre_data, start=1):
        norm_frame = cv2.normalize(frame, None, 0, 255, cv2.NORM_MINMAX)
        uint8_frame = norm_frame.astype(np.uint8)
        rgb_frame = cv2.cvtColor(uint8_frame, cv2.COLOR_GRAY2BGR)
        filename = f"{i:04d}.jpg"
        out_path = os.path.join(output_dir, filename)
        cv2.imwrite(out_path, rgb_frame)

    print(f"✔ {len(pre_data)} slices saved to: {output_dir}")

    # viewer.add_image(pre_data, name='raw', rgb=False)
    
    # napari.run()

    return pre_data, output_dir

# --- SAM2 operation 
def propagate_mask_and_visualize(
    jpg_path,
    predictor,
    label_layer=None,
    obj_id=1,
    seed_planes=None,
    viewer=None
):
    import numpy as np, os, napari
    from napari.layers import Labels as LabelsLayer

    # (0) viewer
    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # (1) label_layer 자동
    if (label_layer is None) or (not hasattr(label_layer, "data")):
        try:
            label_layer = viewer.layers['Labels']
        except KeyError:
            candidates = [lyr for lyr in viewer.layers if isinstance(lyr, LabelsLayer)]
            if not candidates:
                raise RuntimeError(
                    "napari 뷰어에서 Labels 레이어를 찾지 못했습니다. "
                    "viewer.layers['Labels'] 이름을 맞추거나, Labels 레이어를 추가하세요."
                )
            label_layer = candidates[0]
            print(f"⚠️ 'Labels'라는 이름의 레이어가 없어 첫 번째 Labels 레이어('{label_layer.name}')를 사용합니다.")

    # (2) 데이터
    import numpy as np, os
    label_data = np.asarray(label_layer.data)
    if label_data.ndim != 3:
        raise ValueError(f"Labels 데이터는 (Z, Y, X) 3D여야 합니다. 현재 shape={label_data.shape}")
    num_slices = label_data.shape[0]

    # (A) obj_id 존재 슬라이스
    if seed_planes is None:
        seed_planes = np.where(
            (label_data >0 ).reshape(num_slices, -1).any(axis=1)
        )[0].tolist()

    if not seed_planes:
        uniq = np.unique(label_data)
        raise ValueError(
            f"[중단] nonzero 라벨이 없습니다. Labels 유니크 값(일부): {uniq[:20]}"
        )

    print(f"[seed check] nonzero seed, seed_planes (len={len(seed_planes)}): "
          f"{seed_planes[:20]}{' ...' if len(seed_planes)>20 else ''}")

    # (3) SAM2 predictor 초기화
    inference_state = predictor.init_state(video_path=jpg_path)
    predictor.reset_state(inference_state)

    # (B) 씨드 추가
    for fidx in seed_planes:
        manual_mask = (label_data[fidx] >0)
        if manual_mask.any():
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=fidx,
                obj_id=obj_id,
                mask=manual_mask,
            )
            print(f"✔ Added seed @ z={fidx} (px={int(manual_mask.sum())})")
        else:
            print(f"⚠️ z={fidx}에 obj_id={obj_id} 마스크가 비어 있음 → 건너뜀")

    # (4) 전파
    center_seed = seed_planes[len(seed_planes)//2]
    print(f"🔁 Propagating from center z={center_seed}, seeds={len(seed_planes)}")

    video_segments = {}
    for rev in (False, True):
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=center_seed, reverse=rev
        ):
            per_obj_output_mask = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
            if out_frame_idx in video_segments:
                for oid, mask in per_obj_output_mask.items():
                    if oid in video_segments[out_frame_idx]:
                        video_segments[out_frame_idx][oid] |= mask
                    else:
                        video_segments[out_frame_idx][oid] = mask
            else:
                video_segments[out_frame_idx] = per_obj_output_mask

    if len(video_segments) == 0:
        raise RuntimeError("No masks were propagated. Check your seed mask or predictor output.")

    # (5) 라벨 스택 생성
    sample_mask = np.squeeze(next(iter(next(iter(video_segments.values())).values())))
    height, width = sample_mask.shape
    label_stack = np.zeros((num_slices, height, width), dtype=np.uint8)

    for fidx, per_obj_mask in video_segments.items():
        if obj_id in per_obj_mask:
            mask_i = np.squeeze(per_obj_mask[obj_id])
            label_stack[fidx][mask_i > 0] = obj_id

    # # (6) 작업 끝나면 jpg_path 폴더 삭제
    # try:
    #     shutil.rmtree(jpg_path)
    #     print(f"🧹 Removed temporary folder: {jpg_path}")
    # except Exception as e:
    #     print(f"⚠️ Failed to remove {jpg_path}: {e}")

    return label_stack, video_segments

# --- Post-process 
def finalize_mask_3d(label_stack,
                     obj_id=1,
                     suffix=None,                # 🆕 'ooplasm' / 'inner' / 'outer' / 기타
                     return_as_label_id=True):
    
    mask = (label_stack == obj_id)
    if not mask.any():
        raise ValueError(f" obj_id={obj_id} 마스크가 비어 있습니다.")

    # --- preserve the largest object using watershed ---
    dist = ndi.distance_transform_edt(mask)
    thr = 0.3 * dist.max()
    core = dist > thr                      # seed mask
    markers, _ = ndi.label(core)           # 여러 seed 자동 분리
    lbl = segmentation.watershed(-dist, markers, mask=mask)
    mask = lbl == np.bincount(lbl.ravel())[1:].argmax()+1   

    # --- morphological operation (varies with target)
    s = str(suffix).lower()

    config = {
        'pb':         (3, ('opening', 'closing')),
        'ooplasm':    (5, ('opening', 'closing')),
    }
    size, ops = config.get(s, (5, ('closing', 'opening'))) # default
    print(f"[morphology] suffix='{s}', size={size}, ops={ops}")

    selem = morphology.ball(size)
    for op in ops:
        mask = getattr(morphology, f"binary_{op}")(mask, selem)

    dist = ndi.distance_transform_edt(mask)
    dist_smooth = ndi.gaussian_filter(dist.astype(float), sigma=2)
    mask = dist_smooth > 1

    if return_as_label_id:
        out = np.zeros_like(mask, dtype=np.uint8)
        out[mask] = np.uint8(obj_id)
        return out
    else:
        return mask.astype(np.uint8)

# --- Run SAM2 segmentation and post-process for multiple label layers and build a combined 3D mask stack
def process_main(
    jpg_path,
    predictor,
    obj_id=1,
    viewer=None,
    layer_name_keyword="Labels",   # 이 문자열을 포함한 napari Labels 레이어만 대상
    # --- pb 전처리용 외부 의존성 ---
    pre_data=None,                 # 원본/전처리할 볼륨 (Z,Y,X)
    file_path=None,                # jpg_for_sam2 에 전달할 원본 파일 경로
    jpg_for_sam2=None              # callable: (file_path, pre_data) -> (pre_data, output_dir)
):

    if viewer is None:
        try:
            viewer = napari.current_viewer()
        except Exception:
            viewer = napari.Viewer()

    # --- 대상 레이어 수집 ---
    layers = [lyr for lyr in viewer.layers
              if isinstance(lyr, LabelsLayer) and (layer_name_keyword.lower() in lyr.name.lower())]
    if not layers:
        raise RuntimeError(f"이름에 '{layer_name_keyword}'가 포함된 napari Labels 레이어를 찾지 못했습니다.")

    def extract_suffix(name, fallback_idx):
        # 예: 'Labels(inner)' -> 'inner', 'Labels-outer' -> 'outer'
        m = re.search(r'Labels\s*[\(\-_\s]*([^)]+)[\)]*', name, re.IGNORECASE)
        if m:
            suf = m.group(1).strip()
            suf = re.sub(r'\W+', '_', suf)
            if suf:
                return suf
        return f"labels{fallback_idx}"

    # --- 처리 순서: ooplasm -> inner -> outer -> pb (대소문자 무시) ---
    priority = {"ooplasm": 0, "inner": 1, "outer": 2, "pb": 3}
    indexed = []
    for idx, lyr in enumerate(layers, start=1):
        suf = extract_suffix(lyr.name, idx)
        prio = priority.get(suf.lower(), 999)  # 지정 외 항목은 맨 뒤
        indexed.append((prio, suf, lyr))
    indexed.sort(key=lambda x: (x[0], x[1].lower()))

    predicted_labels = {'per_layer': {}, 'order': []}

    for prio, suffix, lyr in indexed:
        suffix_norm = suffix.lower()
        print(f"\n===== Processing: '{lyr.name}' → suffix='{suffix_norm}' (prio={prio}) =====")
        # pb가 아닌 경우: 기존 파이프라인
        if suffix_norm != "pb":
            label_stack, _ = propagate_mask_and_visualize(
                jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)

        else:
            # pb 전처리 요건 확인
            have_inner = f"final_mask_inner" in globals()
            have_ooplas = f"final_mask_ooplasm" in globals()
            if not (have_inner and have_ooplas):
                print("⚠️ PB 전처리 불가: inner/ooplasm 결과가 필요합니다. PB는 건너뜁니다.")
                continue
            if (pre_data is None) or (file_path is None) or (jpg_for_sam2 is None):
                print("⚠️ PB 전처리 불가: pre_data/file_path/jpg_for_sam2 가 필요합니다. PB는 건너뜁니다.")
                continue

            # (inner & ~ooplasm) 마스크 계산
            inner_mask = (globals()["final_mask_inner"] > 0)
            ooplasm_mask = (globals()["final_mask_ooplasm"] > 0)
            pvs_mask = (inner_mask & ~ooplasm_mask)

            if pvs_mask.shape != pre_data.shape:
                raise ValueError(f"PB 전처리: core_mask shape {pvs_mask.shape} != pre_data shape {pre_data.shape}")

            # segmented pre_data 생성: 마스크 위치만 보존
            if np.issubdtype(pre_data.dtype, np.bool_):
                segmented_pre = pre_data & pvs_mask
            else:
                segmented_pre = (pre_data * pvs_mask.astype(pre_data.dtype))

            # jpg set 재생성
            _pre_data_for_sam2, pb_jpg_path = jpg_for_sam2(file_path, segmented_pre)

            # SAM2 실행 (pb용)
            label_stack, _ = propagate_mask_and_visualize(
                pb_jpg_path, predictor, lyr, obj_id, None, viewer
            )
            final_mask = finalize_mask_3d(label_stack, suffix=suffix)
            # viewer.add_labels(final_mask, name="mask_PB_raw")
            # viewer.add_image(segmented_pre, name="segmented_pre")

        # 개별 결과 저장/노출
        globals()[f"label_stack_{suffix_norm}"] = label_stack
        globals()[f"final_mask_{suffix_norm}"]  = final_mask
        predicted_labels['per_layer'][suffix_norm] = dict(label_stack=label_stack, final_mask=final_mask)
        predicted_labels['order'].append(suffix_norm)

    print("\n✅ 개별 마스크 생성 완료(처리 순서):", predicted_labels['order'])

    for k in list(globals().keys()):
        if k.startswith("final_mask_"):
            norm = k.lower()
            if norm not in globals():
                globals()[norm] = globals()[k]

    # --- 4개 마스크를 통합한 label stack 생성 ---
    need = ["ooplasm", "inner", "outer", "pb"]
    if all(f"final_mask_{n}" in globals() for n in need):
        ooplasm = (globals()["final_mask_ooplasm"] > 0)
        inner   = (globals()["final_mask_inner"]   > 0)
        outer   = (globals()["final_mask_outer"]   > 0)
        pb      = (globals()["final_mask_pb"]      > 0)

        label_stack_all = np.zeros_like(ooplasm, np.uint8)
        # 우선순위: ooplasm(1) → inner-only(2) → outer-only(3) → pb-only(4)
        label_stack_all[ooplasm] = 1
        label_stack_all[inner & ~ooplasm] = 2
        label_stack_all[outer & ~(ooplasm | inner)] = 3
        label_stack_all[pb] = 4

        globals()["label_stack_all"] = label_stack_all
        viewer.add_labels(label_stack_all, name="mask_ALL")
        print("✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)")
        predicted_labels["label_stack_all"] = label_stack_all
    else:
        missing = [n for n in need if f"final_mask_{n}" not in globals()]
        print(f"⚠️ 통합 레이블(label_stack_all) 생성 불가: 부족한 마스크 = {missing}")

    return predicted_labels



In [ ]:
# --- multi compartment segmentation
predicted_labels = process_main(
    jpg_path=jpg_path,
    predictor=predictor,
    obj_id=1,
    viewer=viewer,
    layer_name_keyword="Labels",   # napari 레이어 이름 필터
    # finalize_kwargs=None,
    pre_data=pre_data,
    file_path=file_path,
    jpg_for_sam2=jpg_for_sam2
)


===== Processing: 'Labels(ooplasm)' → suffix='ooplasm' (prio=0) =====
[seed check] nonzero seed, seed_planes (len=5): [98, 131, 179, 235, 270]


frame loading (JPEG): 100%|██████████| 360/360 [00:10<00:00, 34.70it/s]
E:\python projects\codes\sam2\sam2\sam2_video_predictor.py:786: UserWarning: cannot import name '_C' from 'sam2' (E:\python projects\codes\sam2\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


✔ Added seed @ z=98 (px=5935)
✔ Added seed @ z=131 (px=21940)
✔ Added seed @ z=179 (px=30861)
✔ Added seed @ z=235 (px=22980)
✔ Added seed @ z=270 (px=4723)
🔁 Propagating from center z=179, seeds=5


propagate in video: 100%|██████████| 180/180 [00:10<00:00, 17.20it/s]


[morphology] suffix='ooplasm', size=5, ops=('opening', 'closing')

===== Processing: 'Labels(inner)' → suffix='inner' (prio=1) =====
[seed check] nonzero seed, seed_planes (len=23): [53, 55, 59, 68, 80, 92, 103, 113, 122, 133, 145, 158, 174, 186, 201, 215, 228, 242, 255, 267] ...


...

[morphology] suffix='outer', size=5, ops=('closing', 'opening')

===== Processing: 'Labels(PB)' → suffix='pb' (prio=3) =====
✔ 360 slices saved to: G:\RealData\sam2
[seed check] nonzero seed, seed_planes (len=7): [83, 85, 89, 95, 101, 104, 105]


frame loading (JPEG): 100%|██████████| 360/360 [00:21<00:00, 16.63it/s]


✔ Added seed @ z=83 (px=290)
✔ Added seed @ z=85 (px=650)
✔ Added seed @ z=89 (px=1146)
✔ Added seed @ z=95 (px=1990)
✔ Added seed @ z=101 (px=688)
✔ Added seed @ z=104 (px=472)
✔ Added seed @ z=105 (px=288)
🔁 Propagating from center z=95, seeds=7


propagate in video: 100%|██████████| 96/96 [00:05<00:00, 16.22it/s]


[morphology] suffix='pb', size=3, ops=('opening', 'closing')

✅ 개별 마스크 생성 완료(처리 순서): ['ooplasm', 'inner', 'outer', 'pb']
✅ label_stack_all 생성 (1=ooplasm, 2=pvs, 3=zona, 4=pb)


ooplasm segmentation

In [ ]:
# --- Set file directory
file_path = r""

# # # --- Preprocess
# pre_data, data = preprocess(file_path)
pre_data, output_dir = jpg_for_sam2(file_path, pre_data)

# # # --- Run viewer: preprae preprocessed images 
viewer = napari.Viewer()
viewer.add_image(pre_data, name = 'preprocessed')
viewer.add_image(data, name = 'HT(downsampled)')

# # --- Load labels: one may generate labels if labels = None
load_labels(file_path, viewer) # one may generate labels 

✔ 360 slices saved to: G:\RealData\sam2


Viewer(camera=Camera(center=(0.0, np.float64(134.0), np.float64(179.5)), zoom=np.float64(1.5833333333333333), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(179.0), 1.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=np.float64(10.0)), dims=Dims(ndim=3, ndisplay=2, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(359.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(268.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(359.0), step=np.float64(1.0))), margin_left=(0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0), point=(np.float64(179.0), np.float64(134.0), np.float64(179.0)), last_used=0), grid=GridCanvas(stride=1, shape=

In [ ]:
# after plugging manual labels
label_stack, _ = propagate_mask_and_visualize(
    jpg_path, predictor, None, 1, None, viewer
)

final_mask = finalize_mask_3d(
    label_stack,
    obj_id=1,
    suffix = "outer",
    return_as_label_id=True
)
viewer.add_labels(label_stack, name = "sam2")
viewer.add_labels(final_mask, name = "final")